# 2D RWKV ground-state search

Small transverse-field Ising example using the imported causal 2D RWKV ansatz.

In [5]:
import jax
import jax.numpy as jnp

import jVMC_exp
import jVMC_exp.nets as nets
import jVMC_exp.operator.discrete as op
import jVMC_exp.sampler as sampler
from jVMC_exp.vqs import NQS

jax.config.update("jax_enable_x64", True)

In [6]:
Lx, Ly = 2, 2
L = Lx * Ly
g = 1.0

def site(x, y):
    return y * Lx + x

bonds = set()
for y in range(Ly):
    for x in range(Lx):
        i = site(x, y)
        bonds.add(tuple(sorted((i, site((x + 1) % Lx, y)))))
        bonds.add(tuple(sorted((i, site(x, (y + 1) % Ly)))))

hamiltonian = 0
for i, j in sorted(bonds):
    hamiltonian += -op.SigmaZ(i) * op.SigmaZ(j)
for i in range(L):
    hamiltonian += -g * op.SigmaX(i)

In [7]:
net = nets.LogRWKV2DAutoregressiveJVMC(
    L=L,
    LHilDim=2,
    patch_size=1,
    grid_Lx=Lx,
    grid_Ly=Ly,
    hidden_size=8,
    num_heads=1,
    num_layers=1,
    embedding_size=8,
    flag_phase=True,
    param_dtype=jnp.float32,
    compute_dtype=jnp.float32,
)
psi = NQS(net, (Ly, Lx), batchSize=2**L, seed=1234)
exact_sampler = sampler.ExactSampler(psi)
print("Number of parameters:", psi.numParameters)

Number of parameters: 1744


In [ ]:
loss_function = jVMC_exp.objective_function.Observable(hamiltonian)
stepper = jVMC_exp.stepper.Euler(5e-2)
opt = jVMC_exp.optimizer.MinSR(exact_sampler, psi, diagonalShift=1e-3, pinv_tol=1e-8, full_batched=True)

out = opt.ground_state_search(500, loss_function, stepper)
energy = exact_sampler(hamiltonian).mean
print("Final energy:", energy)